# ML vs LLM: shared-metric comparison on CA prediction

**Goal.** Evaluate traditional machine-learning models (Random Forest, KNN) against LLM persona agents on the **same** prediction task and the **same** metrics.

Both families predict PRCA **group** and **interpersonal** subscale scores (6–30) from the curated persona information tiers (`demos` → `employment` → `geo` → `transit`).

| Metric family | Indicators |
|---|---|
| Score precision | MAE, exact integer match |
| Band agreement | low / moderate / high accuracy |
| Distance from correct | normalized score distance (`|e|/24`), ordinal band distance (0–2) |

Supporting code: [`src/ca_personas/compare_agents.py`](../src/ca_personas/compare_agents.py).

## 1. Setup

Set `LLM_PROVIDER` to `mock` for offline runs, or `ollama` / `openrouter` for live models.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ca_personas.compare_agents import (
    COMPARISON_METRICS,
    run_ml_vs_llm_comparison,
)
from ca_personas.personas import RESEARCH_TIERS

PROLIFIC = ROOT / "data" / "excerpts" / "prolific_excerpt.csv"
QUALTRICS = ROOT / "data" / "excerpts" / "qualtrics_excerpt.csv"
OUT_DIR = ROOT / "outputs" / "ml_vs_llm"

LLM_PROVIDER = os.getenv("CA_LLM_PROVIDER", "mock")
LLM_MODEL = os.getenv("OLLAMA_MODEL") or os.getenv("OPENROUTER_MODEL") or None
TIERS = list(RESEARCH_TIERS)

pd.set_option("display.max_columns", 40)
print("LLM provider:", LLM_PROVIDER, "| tiers:", TIERS)

## 2. Run both agent families

1. **ML** — cross-validated Random Forest + KNN on each tier  
2. **LLM** — persona prompts → model JSON scores/bands  

Both are scored with `evaluate_predictions` / `summarize_errors`.

In [ ]:
result = run_ml_vs_llm_comparison(
    PROLIFIC,
    QUALTRICS,
    tiers=TIERS,
    llm_provider=LLM_PROVIDER,
    llm_model=LLM_MODEL,
    join_how="inner",
    output_dir=OUT_DIR,
)

comparison = result["comparison"]
deltas = result["deltas"]
evaluation = result["evaluation"]

print("Artifacts:")
for k, p in result["artifacts"].items():
    print(f"  {k:12s} -> {p}")

comparison[comparison["tier"] != "all"][
    ["agent", "agent_family", "tier", "n_with_ground_truth"]
    + [c for c in COMPARISON_METRICS if c in comparison.columns][:6]
]

## 3. Head-to-head metric table

Lower is better for MAE / distance metrics; higher is better for exact/band accuracy.

In [ ]:
tier_compare = comparison[comparison["tier"] != "all"].copy()
display_cols = [
    "agent",
    "agent_family",
    "tier",
    "mae_group",
    "mae_interpersonal",
    "exact_acc_group",
    "exact_acc_interpersonal",
    "band_acc_group",
    "band_acc_interpersonal",
    "mean_norm_score_distance_group",
    "mean_band_distance_group",
]
tier_compare[[c for c in display_cols if c in tier_compare.columns]].sort_values(
    ["tier", "agent_family", "mae_group"]
)

## 4. Visual comparison by tier

In [ ]:
metrics_to_plot = [
    ("mae_group", "MAE — Group CA (↓ better)"),
    ("mae_interpersonal", "MAE — Interpersonal CA (↓ better)"),
    ("band_acc_group", "Band accuracy — Group (↑ better)"),
    ("mean_norm_score_distance_group", "Norm. score distance — Group (↓ better)"),
]

agents = sorted(tier_compare["agent"].unique())
colors = {
    a: ("#C5050C" if str(a).startswith("ml:") else "#333333")
    for a in agents
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.ravel()
for ax, (metric, title) in zip(axes, metrics_to_plot):
    for agent in agents:
        frame = tier_compare[tier_compare["agent"] == agent].sort_values("tier")
        if metric not in frame.columns:
            continue
        ax.plot(
            frame["tier"],
            frame[metric],
            marker="o",
            label=agent,
            color=colors.get(agent),
        )
    ax.set_title(title)
    ax.set_xlabel("Persona / feature tier")
    ax.tick_params(axis="x", rotation=20)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=min(4, len(labels)), frameon=False)
fig.suptitle("ML vs LLM performance across information tiers", y=1.02)
fig.tight_layout()
plt.show()

## 5. Deltas vs best ML baseline

For error/distance metrics, **negative** `delta_vs_best_ml` means the agent beats the strongest ML model on that tier. For accuracy metrics, **positive** deltas are better.

In [ ]:
delta_view = deltas.copy()
delta_view.head(24)

In [ ]:
mae_delta = deltas[deltas["metric"] == "mae_group"]
fig, ax = plt.subplots(figsize=(9, 4.5))
for family, frame in mae_delta.groupby("agent_family"):
    ax.scatter(frame["tier"], frame["delta_vs_best_ml"], label=family, s=70, alpha=0.85)
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Δ MAE group vs best ML")
ax.set_xlabel("Tier")
ax.set_title("Group-CA MAE relative to best ML baseline (negative = better than ML)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 6. Per-participant residuals (optional deep dive)

Inspect whether LLM misses concentrate on the same participants as ML.

In [ ]:
resid_cols = [
    c
    for c in [
        "participant_id",
        "tier",
        "agent",
        "agent_family",
        "pred_group_ca",
        "gt_group_ca",
        "abs_error_group",
        "band_match_group",
        "band_distance_group",
        "norm_score_distance_group",
    ]
    if c in evaluation.columns
]
evaluation[resid_cols].sort_values(["participant_id", "tier", "agent_family"]).head(24)

## 7. Interpretation checklist

1. Does any LLM tier beat RF/KNN on **MAE** or **normalized score distance**?
2. Does the LLM win on **band accuracy** even when exact scores miss (near-miss viability)?
3. As tiers add employment / geo / transit, do ML and LLM improve at the same rate?
4. Replace `mock` with Ollama/OpenRouter for the manuscript comparison; the metric definitions stay fixed.

```bash
CA_LLM_PROVIDER=ollama OLLAMA_MODEL=llama3.2 \
  jupyter nbconvert --to notebook --execute notebooks/ml_vs_llm_comparison.ipynb
```